This notebook is intended to better understand the PhenoGraph classify function.

In [2]:
# Import packages
from typing import Optional
import phenograph
import scanpy as sc
from anndata import AnnData
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib
import os
import pandas as pd
import numpy as np
from scipy.stats import entropy
import subprocess
#import palantir
import sys
import random
from collections import OrderedDict
import re
from itertools import chain
import warnings
sys.path.append('/lila/home/forsythb/anaconda3/envs/scrna/lib/python3.8/site-packages')
import harmony
import numpy as np
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
import scipy.sparse as sp
from sklearn.preprocessing import normalize

findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Lato'] not found. Falling back to DejaVu Sans.


In [3]:
%matplotlib inline

In [4]:
adata_patient_organoid = sc.read_h5ad('/data/chanjlab/forsythb/adata_patient_organoid_2.h5ad')

In [5]:
aug_mat = pd.DataFrame(adata_patient_organoid.obsm['aug_mat'], index=adata_patient_organoid.obs.index, columns = adata_patient_organoid.obs.index)

In [6]:
ind = adata_patient_organoid.obs['Cell State'] != 'nan'
adata_patient_organoid = adata_patient_organoid[ind,:]
aug_mat = aug_mat.loc[ind,ind]

In [7]:
###OLD

In [8]:
# Select cells with labeled cell state (excluding 'NA')
labeled_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'NA']

# Get unique cell types/categories
cell_types = labeled_cells.obs['Cell State'].unique()

# Initialize an empty list to store training data for each cell type
train_data = []

# Iterate over cell types and extract 'aug_mat' for each
for cell_type in cell_types:
    # Filter cells of the current cell type
    cells_of_type = labeled_cells[labeled_cells.obs['Cell State'] == cell_type]
    
    # Extract 'aug_mat' for these cells
    aug_mat_data = cells_of_type.obsm['aug_mat'].toarray()  # Assuming sparse matrix to array
    
    # Append 'aug_mat' data to the list
    train_data.append(aug_mat_data)

In [9]:
# # Select cells with labeled cell state (excluding 'nan') for test data
# test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'nan']

# # Extract 'aug_mat' for test cells
# test_data = test_cells.obsm['aug_mat'].toarray()


# Select cells with labeled cell state (excluding 'nan') for test data
test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'nan']

# Exclude cells with 'KG146M' in observation names
test_cells = test_cells[~test_cells.obs_names.str.contains('_KG146M')]

# Extract 'aug_mat' for test cells
test_data = test_cells.obsm['aug_mat'].toarray()

In [10]:
test_cells.obs_names

Index(['120703408859571_OKG146P_Base', '120703423732149_OKG146P_Base',
       '120703423986909_OKG146P_Base', '120703436417774_OKG146P_Base',
       '120718456404405_OKG146P_Base', '120726897154790_OKG146P_Base',
       '120726943063347_OKG146P_Base', '120728776760118_OKG146P_Base',
       '120728803268403_OKG146P_Base', '120728822889307_OKG146P_Base',
       ...
       '241097696495478_OKG146Li_ENAFI', '241097696536940_OKG146Li_ENAFI',
       '241109193152427_OKG146Li_ENAFI', '241109193570547_OKG146Li_ENAFI',
       '241109193845101_OKG146Li_ENAFI', '241109208263404_OKG146Li_ENAFI',
       '241109208558948_OKG146Li_ENAFI', '241176048221470_OKG146Li_ENAFI',
       '241176048777525_OKG146Li_ENAFI', '241184516820315_OKG146Li_ENAFI'],
      dtype='object', length=9092)

In [11]:
ct_metacluster_preds_dm = phenograph.classify(train_data, test_data, k=30, metric='euclidean')

Finding 30 nearest neighbors using minkowski metric and 'auto' algorithm


KeyboardInterrupt: 

In [ ]:
###OLD

In [ ]:
import numpy as np
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
import scipy.sparse as sp
from sklearn.preprocessing import normalize


def random_walk_probabilities(A, labels):

    if labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()
        D = sp.diags(A.sum(axis=1).flatten(), [0], shape=A.shape, format='csr')
        L = D - A
        seeds = labels != 0
        Lu = L[np.invert(seeds), :]  # unlabeled rows
        Lu = Lu.tocsc()[:, np.invert(seeds)]  # unlabeled columns
        # Check that Lu has the right size
        if not all([t == sum(np.invert(seeds)) for t in Lu.shape]):
            raise IndexError("Lu should be square and match size of test data")
        BT = L[np.invert(seeds), :]  # unlabeled rows
        BT = BT.tocsc()[:, seeds]  # labeled columns
        if not (sum(np.invert(seeds)), sum(seeds)) == BT.shape:
            raise IndexError("BT size is incorrect")
        # Matrix representation of labels
        i, j, s = [], [], []
        classes = np.unique(labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(labels[seeds] == k)[0])
            j.extend(np.tile(k, sum(seeds == k)))
            s.extend(np.tile(1, sum(seeds == k)))
        i = np.arange(seeds.sum())
        j = labels[seeds] - 1
        s = np.ones((seeds.sum(), ))
        M = sp.coo_matrix((s, (i, j)), shape=(seeds.sum(), len(classes))).tocsc()
        # P = sp.linalg.spsolve(Lu.tocsc(), -BT.dot(M))
        # Use iterative solver
        B = -BT.dot(M)
        vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]
        warnings = [x[1] for x in vals]
        if any(warnings):
            print("Warning: iterative solver failed to converge in at least one case", flush=True)
        P = normalize(np.vstack(tuple((x[0] for x in vals))).T, norm='l1')

    else:
        D = np.diag(np.sum(A, axis=1))
        L = D - A  # graph laplacian
        seeds = np.array([e != 0 for e in labels], dtype=bool)
        Lu = L[np.invert(seeds), np.invert(seeds)]  # unlabeled rows, unlabeled cols
        BT = L[np.invert(seeds), seeds]  # unlabeled rows, labeled cols
        classes = np.unique(labels[labels > 0])
        M = np.zeros((seeds.sum(), len(classes)))
        for k in classes:
            M[labels[seeds] == k, k] = 1
        P = np.linalg.lstsq(Lu, np.dot(-BT, M))[0]

    return P

In [ ]:
def preprocess(train, test):
    labels = np.zeros((test.shape[0], ), dtype=int)
    data = test
    for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data = np.append(data, examples, axis=0)
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    return data, labels

In [ ]:
# Select cells with labeled cell state (excluding 'NA')
ind = adata_patient_organoid.obs['Cell State'] != 'NA'
labeled_cells = adata_patient_organoid[ind,:]


# Get unique cell types/categories
cell_types = labeled_cells.obs['Cell State'].unique()

# Initialize an empty list to store training data for each cell type
train_data = []
train_labels = []
ct_codes = []

# Iterate over cell types and extract 'aug_mat' for each
for i,cell_type in enumerate(cell_types):
    # Filter cells of the current cell type
    cells_of_type = labeled_cells[labeled_cells.obs['Cell State'] == cell_type]
    
    # Extract 'aug_mat' for these cells
    pc_data = cells_of_type.obsm['X_pca']  # Assuming sparse matrix to array
    
    # Append 'aug_mat' data to the list
    train_data.append(pc_data)
    train_labels.append(cells_of_type.obs.index.to_series())   
    ct_codes += [i+1]*pc_data.shape[0]

In [ ]:
train_labels = pd.concat(train_labels).index

In [ ]:
# # Select cells with labeled cell state (excluding 'nan') for test data
test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] == 'NA']

# # Extract 'aug_mat' for test cells
test_data = test_cells.obsm['X_pca']
test_labels = test_cells.obs.index

# Select cells with labeled cell state (excluding 'nan') for test data
# test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'nan']

# Exclude cells with 'KG146M' in observation names
# test_cells = test_cells[~test_cells.obs_names.str.contains('_KG146M')]

# Extract 'aug_mat' for test cells
# test_data = test_cells.obsm['aug_mat'].toarray()

In [ ]:
labels = pd.Index(test_labels.tolist() +  train_labels.tolist())

In [ ]:
from scipy.sparse import coo_matrix

In [ ]:
aug_all = coo_matrix(aug_mat.loc[labels,labels].values )

In [ ]:
ct_codes = np.array([0]*len(test_labels) + ct_codes)

In [ ]:
data, labels = preprocess(train_data, test_data)

In [ ]:
data.shape

In [ ]:
labels

In [ ]:
def create_graph(data, k=30, metric='euclidean', n_jobs=-1):
    # def _kernel(dxy, sigma=1):
    #     return np.exp(-dxy ** 2 / sigma)

    _, idx = find_neighbors(data, k=k, metric=metric, n_jobs=n_jobs)
    # affinities = np.apply_along_axis(lambda x: _kernel(x, x.std()), axis=1, arr=d)
    # n, k = idx.shape
    # i = [np.tile(x, (k, )) for x in range(n)]
    # i = np.concatenate(np.array(i))
    # j = np.concatenate(idx)
    # s = np.concatenate(affinities)
    # graph = sp.coo_matrix((s, (i, j)), shape=(n, n)).tocsr()
    # graph = (graph + graph.transpose()).multiply(.5)
    graph = neighbor_graph(jaccard_kernel, {'idx': idx})
    # make symmetric
    # graph = (graph + graph.transpose()).multiply(.5)
    return graph

In [ ]:
create_graph(aug_all[:50,:10])

In [ ]:
P = random_walk_probabilities(aug_all, ct_codes)

In [ ]:
P

In [336]:
pval_df = pd.DataFrame(P,
                           index = test_cells.obs_names,
                           columns = cell_types)

In [338]:
pval_df
pval_df.to_csv('/data/chanjlab/forsythb/phenograph_classification_6.csv', index=True)

In [107]:
pval_df = pd.DataFrame(ct_metacluster_preds_dm[1],
                           index = test_cells.obs_names,
                           columns = cell_types)

In [108]:
pval_df.to_csv('/data/chanjlab/forsythb/phenograph_classification_5.csv', index=True)

In [147]:
pval_df = pd.read_csv('/data/chanjlab/forsythb/phenograph_classification_5.csv', index_col=0)

In [211]:
pval_df

,ISC-like,SCC,Fetal Progenitor,Enterocyte-like,TA-like,Injury Repair,Early NET,Goblet-like
120703408859571_OKG146P_Base,0.010385,0.224508,0.000150,0.761122,0.001263,0.000579,0.000055,0.001938
120703423732149_OKG146P_Base,0.010926,0.220312,0.000147,0.764036,0.001268,0.000645,0.000095,0.002573
120703423986909_OKG146P_Base,0.010368,0.226601,0.000148,0.759042,0.001260,0.000591,0.000055,0.001936
120703436417774_OKG146P_Base,0.010426,0.220445,0.000146,0.765140,0.001271,0.000573,0.000055,0.001944
120718456404405_OKG146P_Base,0.010349,0.227853,0.000148,0.757818,0.001258,0.000588,0.000055,0.001932
...,...,...,...,...,...,...,...,...
241109220584862_KG146M,0.001183,0.918078,0.000041,0.080290,0.000133,0.000065,0.000006,0.000203
241176048225691_KG146M,0.362695,0.123493,0.000082,0.445196,0.062957,0.003994,0.000032,0.001550
241176061270774_KG146M,0.022700,0.040493,0.000027,0.178361,0.000348,0.000302,0.000011,0.757757
241184516782309_KG146M,0.001727,0.510383,0.359469,0.127784,0.000211,0.000096,0.000009,0.000321


In [152]:
tmp = pd.concat([pval_df, adata_patient_organoid.obs.loc[:,'PhenoGraph Classification']], axis=1)

In [161]:
tmp = tmp.loc[~tmp.index.str.contains('_KG146P'),:]

In [168]:
tmp2 = pd.concat([tmp.iloc[:,:-1].idxmax(axis=1), tmp.iloc[:,-1]], axis=1)

In [171]:
tmp3 = tmp2.loc[tmp2.iloc[:,-1]!='nan',:]

In [175]:
(tmp3.iloc[:,0] == tmp3.iloc[:,1]).value_counts()

False    5891
True     3201
Name: count, dtype: int64

In [178]:
from scipy.stats import entropy

In [182]:
H = tmp.iloc[:,:-1].apply(entropy, axis=1)

In [186]:
plot_df = pd.concat([H, tmp3.iloc[:,-1]], axis=1)
plot_df.columns=['H','PredictedCellType']


In [205]:
plot_df.PredictedCellType = plot_df.PredictedCellType.astype(str)

In [207]:
plot_df.groupby('PredictedCellType').mean()

,H
PredictedCellType,
Early NET,0.654188
Enterocyte-like,0.618896
Fetal Progenitor,0.615837
Goblet-like,0.648637
ISC-like,0.619552
Injury Repair,0.619171
SCC,0.616435
TA-like,0.625891
nan,0.820508


In [192]:
sns.boxplot(plot_df, x='PredictedCellType', y = 'H')

/home/forsythb/.local/lib/python3.9/site-packages/seaborn/_oldcore.py:1498: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(vector):
/home/forsythb/.local/lib/python3.9/site-packages/seaborn/categorical.py:641: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped_vals = vals.groupby(grouper)


<Axes: xlabel='PredictedCellType', ylabel='H'>

findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.
findfont: Font family 'Bitstream Vera Sans' not found.


In [130]:
test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'nan']

In [117]:
selected_cell_types = ['P(ISC-like)', 'P(SCC)', 'P(Fetal Progenitor)', 'P(Enterocyte-like)',
                        'P(TA-like)', 'P(Injury Repair)', 'P(Early NET)', 'P(Goblet-like)']

# Filter labeled_cells.obs based on desired observations
filtered_obs = test_cells.obs[selected_cell_types]

# Create a new dataframe
new_dataframe = pd.DataFrame(filtered_obs.values, index=test_cells.obs.index, columns=selected_cell_types)

In [118]:
new_dataframe

,P(ISC-like),P(SCC),P(Fetal Progenitor),P(Enterocyte-like),P(TA-like),P(Injury Repair),P(Early NET),P(Goblet-like)
120703408859571_OKG146P_Base,0.276070,0.0,0.002401,0.253181,0.405188,0.024051,0.002132,0.036976
120703423732149_OKG146P_Base,0.235303,0.0,0.021353,0.176145,0.004448,0.060198,0.130694,0.371859
120703423986909_OKG146P_Base,0.233067,0.0,0.003073,0.622498,0.042236,0.037871,0.004873,0.056382
120703436417774_OKG146P_Base,0.139115,0.0,0.000126,0.846154,0.009440,0.001682,0.000208,0.003276
120718456404405_OKG146P_Base,0.256664,0.0,0.003672,0.373109,0.234635,0.058768,0.007674,0.065478
...,...,...,...,...,...,...,...,...
241109220584862_KG146M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
241176048225691_KG146M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
241176061270774_KG146M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
241184516782309_KG146M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [119]:
test_cells.obs_names

Index(['120703408859571_OKG146P_Base', '120703423732149_OKG146P_Base',
       '120703423986909_OKG146P_Base', '120703436417774_OKG146P_Base',
       '120718456404405_OKG146P_Base', '120726897154790_OKG146P_Base',
       '120726943063347_OKG146P_Base', '120728776760118_OKG146P_Base',
       '120728803268403_OKG146P_Base', '120728822889307_OKG146P_Base',
       ...
       '241046943394526_KG146M', '241097677924214_KG146M',
       '241109193321699_KG146M', '241109193414900_KG146M',
       '241109208586659_KG146M', '241109220584862_KG146M',
       '241176048225691_KG146M', '241176061270774_KG146M',
       '241184516782309_KG146M', '241184517057252_KG146M'],
      dtype='object', length=10396)

In [88]:
def preprocess(train, test):
    labels = np.zeros((test.shape[0], ), dtype=int)
    data = test
    for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data = np.append(data, examples, axis=0)
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    return data, labels

In [95]:
import numpy as np
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
import scipy.sparse as sp
from sklearn.preprocessing import normalize


def random_walk_probabilities(A, labels):

    if labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()
        D = sp.diags(A.sum(axis=1).flatten(), [0], shape=A.shape, format='csr')
        L = D - A
        seeds = labels != 0
        Lu = L[np.invert(seeds), :]  # unlabeled rows
        Lu = Lu.tocsc()[:, np.invert(seeds)]  # unlabeled columns
        # Check that Lu has the right size
        if not all([t == sum(np.invert(seeds)) for t in Lu.shape]):
            raise IndexError("Lu should be square and match size of test data")
        BT = L[np.invert(seeds), :]  # unlabeled rows
        BT = BT.tocsc()[:, seeds]  # labeled columns
        if not (sum(np.invert(seeds)), sum(seeds)) == BT.shape:
            raise IndexError("BT size is incorrect")
        # Matrix representation of labels
        i, j, s = [], [], []
        classes = np.unique(labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(labels[seeds] == k)[0])
            j.extend(np.tile(k, sum(seeds == k)))
            s.extend(np.tile(1, sum(seeds == k)))
        i = np.arange(seeds.sum())
        j = labels[seeds] - 1
        s = np.ones((seeds.sum(), ))
        M = sp.coo_matrix((s, (i, j)), shape=(seeds.sum(), len(classes))).tocsc()
        # P = sp.linalg.spsolve(Lu.tocsc(), -BT.dot(M))
        # Use iterative solver
        B = -BT.dot(M)
        vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]
        warnings = [x[1] for x in vals]
        if any(warnings):
            print("Warning: iterative solver failed to converge in at least one case", flush=True)
        P = normalize(np.vstack(tuple((x[0] for x in vals))).T, norm='l1')

    else:
        D = np.diag(np.sum(A, axis=1))
        L = D - A  # graph laplacian
        seeds = np.array([e != 0 for e in labels], dtype=bool)
        Lu = L[seeds, :][:, seeds]  # labeled rows, labeled cols
        BT = L[~seeds, :][:, seeds]  # unlabeled rows, labeled cols
        classes = np.unique(labels[labels > 0])
        M = np.zeros((seeds.sum(), len(classes)))
        for k in classes:
            M[labels[seeds] == k, k] = 1
        P = np.linalg.lstsq(Lu, np.dot(-BT, M))[0]

    return P

In [96]:
def classify(train, test, k=30, metric='euclidean', n_jobs=-1):
    """
    Semi-supervised classification by random walks on a graph
    :param train: list of numpy arrays. Each array has a row for each class observation
    :param test: numpy array of unclassified data
    :return c: class assignment for each row in test
    :return P: class probabilities for each row in test
    """
    data, labels = preprocess(train, test)
    # Build graph
    A = adata_patient_organoid.obsm['aug_mat']
    P = random_walk_probabilities(A, labels)
    c = np.argmax(P, axis=1)
    return c, P

In [97]:
n_neighbors = 60
ct_metacluster_preds_dm = classify(train_data, test_data, k=n_neighbors, metric='euclidean')

IndexError: boolean index did not match indexed array along dimension 0; dimension is 11324 but corresponding boolean dimension is 11700

In [72]:
# Select cells with labeled cell state (excluding 'NA')
labeled_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'NA']

# Get unique cell types/categories
cell_types = labeled_cells.obs['Cell State'].unique()

# Initialize an empty list to store training data for each cell type
train_data = []

# Iterate over cell types and extract 'aug_mat' and class label for each
for cell_type in cell_types:
    # Filter cells of the current cell type
    cells_of_type = labeled_cells[labeled_cells.obs['Cell State'] == cell_type]
    
    # Extract 'aug_mat' for these cells
    aug_mat_data = cells_of_type.obsm['aug_mat'].toarray()  # Assuming sparse matrix to array
    
    # Create a tuple with 'aug_mat' data and class label
    cell_type_data = (aug_mat_data, cell_type)
    
    # Append the tuple to the list
    train_data.append(cell_type_data)

In [75]:
train_data[0]

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]),
 'ISC-like')

In [37]:
adata_patient_organoid.obs['Cell State'].replace('nan', np.nan, inplace=True)
adata_patient_organoid.obs.dropna(subset=['Cell State'], inplace=True)
train_adata = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'NA']
test_adata = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] == 'NA']

In [42]:
cell_types = sorted(train_adata.obs['Cell State'].cat.categories)

dm_merge = pd.DataFrame(test_adata.obsm['aug_mat'], index=test_adata.obs.index).loc[:, :]

train = np.empty((len(cell_types),),dtype=object)

for c, cell_type in enumerate(cell_types):
    labels = train_adata.obs.index[train_adata.obs['Cell State'] == cell_type]
    #print(f'Cell Type: {cell_type}, Labels: {labels}')
    ind = [train_adata.obs.index.get_loc(i) for i in labels]
    train[c] = dm_merge.iloc[ind, :]
    
data = test_adata

In [45]:
def classify(data, labels, train, test, k=30, metric='euclidean', n_jobs=-1):
    """
    Semi-supervised classification by random walks on a graph
    :param data: Data matrix
    :param labels: Labels associated with the data
    :param adata_patient_organoid: AnnData object with augmented matrix
    :param k: Number of neighbors for affinity matrix
    :param metric: Distance metric for affinity matrix
    :param n_jobs: Number of parallel jobs for affinity matrix computation
    :return c: class assignment for each row in test
    :return P: class probabilities for each row in test
    """
    # Use augmented affinity matrix from adata_patient_organoid.obsm['aug_mat']
    A = train_adata.obsm['aug_mat']

    # Ensure that the affinity matrix has the correct shape
    if A.shape[0] != A.shape[1] or A.shape[0] != len(labels):
        raise ValueError("Mismatch in the shape of the affinity matrix and labels.")

    # Compute random walk probabilities
    P = random_walk_probabilities(A, labels)

    # Class assignment based on the highest probability
    c = np.argmax(P, axis=1)

    return c, P

In [46]:
c, P = classify(data, labels, train, data, k=30, metric='euclidean', n_jobs=-1)

ValueError: Mismatch in the shape of the affinity matrix and labels.

In [20]:
# Separate training and testing sets based on 'Cell State' column
train_adata = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] == 'nan']
test_adata = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] != 'nan']

In [23]:
train_adata.obs['Cell State']

230677806275293_KG146P    nan
122359118682484_KG146P    nan
239476046976795_KG146P    nan
226760915340077_KG146P    nan
169156994812654_KG146P    nan
                         ... 
165190304708846_KG146P    nan
161384145836979_KG146P    nan
200897607945508_KG146P    nan
125033721486630_KG146P    nan
200444891487139_KG146P    nan
Name: Cell State, Length: 928, dtype: category
Categories (1, object): ['nan']

In [24]:
test_adata.obs['Cell State']

120703408859571_OKG146P_Base             NA
120703423732149_OKG146P_Base             NA
120703423986909_OKG146P_Base             NA
120703436417774_OKG146P_Base             NA
120718456404405_OKG146P_Base             NA
                                   ...     
241109220584862_KG146M                  SCC
241176048225691_KG146M             ISC-like
241176061270774_KG146M          Goblet-like
241184516782309_KG146M                  SCC
241184517057252_KG146M              TA-like
Name: Cell State, Length: 10396, dtype: category
Categories (9, object): ['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', ..., 'Injury Repair', 'NA', 'SCC', 'TA-like']

In [35]:
adata_patient_organoid.obs['Cell State'].value_counts()

Cell State
NA                  9092
nan                  928
SCC                  280
ISC-like             252
Injury Repair        230
Enterocyte-like      223
Fetal Progenitor     106
TA-like               80
Goblet-like           71
Early NET             62
Name: count, dtype: int64

In [34]:
adata_patient_organoid.obs['PhenoGraph Classification'].value_counts()

PhenoGraph Classification
Enterocyte-like     3201
nan                 2232
TA-like             1913
ISC-like            1770
Injury Repair       1160
Goblet-like          371
Fetal Progenitor     318
SCC                  295
Early NET             64
Name: count, dtype: int64

In [32]:
adata_patient_organoid

AnnData object with n_obs × n_vars = 11324 × 20157
    obs: 'n_counts', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'original_total_counts', 'log10_original_total_counts', 'original_mito_counts', 'log10_original_mito_counts', 'Sample ID', 'emptyDrops_Total', 'emptyDrops_LogProb', 'emptyDrops_PValue', 'emptyDrops_FDR', 'latent_cell_probability', 'PhenoGraph_clusters', 'Smilie Cell Type', 'log10_n_counts', 'frac_counts_gt1_reads', 'n_reads', 'n_reads_per_count', 'n_counts_vs_ED_LogProb', 'n_counts_vs_n_genes_by_counts', 'n_flagged_metrics', 'DC 1', 'DC 2', 'DC 3', 'DC 4', 'DC 5', 'DC 6', 'DC 7', 'Palantir Diff. Potential', 'Palantir Pseudotime', 'Combined Fetal Signature', 'First Tri

In [27]:
import numpy as np

def preprocess(train, test):
    unique_labels = np.unique(test.obs['Cell State'])  # Get unique category labels from test_adata
    labels = np.zeros((test.shape[0],), dtype=int)
    data = test.X  # Assuming X contains the examples in AnnData object

    for c, label in enumerate(unique_labels):
        indices = np.where(test.obs['Cell State'] == label)[0]
        labels[indices] = c + 1
        data = np.append(data, test.X[indices], axis=0)

    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")

    return data, labels

In [29]:
import numpy as np

def preprocess(train, test):
    unique_labels = np.unique(test.obs['Cell State'])  # Get unique category labels from test_adata
    labels = np.zeros((test.shape[0],), dtype=int)
    data = test.X  # Assuming X contains the examples in AnnData object

    for c, label in enumerate(unique_labels):
        indices = np.where(test.obs['Cell State'] == label)[0]
        
        if len(indices) > 0:
            labels[indices] = c + 1
            data = np.append(data, test.X[indices], axis=0)

    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")

    return data, labels

In [16]:
import numpy as np

def preprocess(train, test):
    labels_list = []
    data_list = []
    print("Starting preprocessing...")
    
    # Find the maximum number of features in train
    max_features = max(examples.shape[1] for examples in train.X)
    
    for c, examples in enumerate(train.X):
        print(f"Processing class {c+1} with {examples.shape[0]} examples...")
        labels_list.append(np.tile(c+1, (examples.shape[0],)))
        
        # Flatten and pad examples with zeros
        flattened_examples = examples.reshape(-1)
        padded_flattened_examples = np.pad(flattened_examples, (0, max_features * examples.shape[0] - len(flattened_examples)), mode='constant')
        data_list.append(padded_flattened_examples)
    
    labels = np.concatenate(labels_list)
    data = np.concatenate(data_list, axis=0)
    
    # Reshape data back to its original form
    data = data.reshape(-1, max_features)
    
    print("Preprocessing complete.")
    
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if np.sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    
    print("Validation complete.")
    return data, labels


In [30]:
# Preprocess the data
data, labels = preprocess(train_adata, test_adata)

ValueError: zero-dimensional arrays cannot be concatenated

In [ ]:
unique_labels, counts = np.unique(labels, return_counts=True)
print("Unique Labels:", unique_labels)
print("Label Counts:", counts)

In [9]:
adata_patient = adata_patient_organoid[adata_patient_organoid.obs['Media'] == 'hISC']
adata_patient.obs['Media']

120703409150835_OKG146P_ENAFI     hISC
120703424322803_OKG146P_ENAFI     hISC
120703455250268_OKG146P_ENAFI     hISC
120703455546804_OKG146P_ENAFI     hISC
120703455717291_OKG146P_ENAFI     hISC
                                  ... 
241109208263404_OKG146Li_ENAFI    hISC
241109208558948_OKG146Li_ENAFI    hISC
241176048221470_OKG146Li_ENAFI    hISC
241176048777525_OKG146Li_ENAFI    hISC
241184516820315_OKG146Li_ENAFI    hISC
Name: Media, Length: 4762, dtype: category
Categories (1, object): ['hISC']

In [3]:
adata_patient_organoid.obsm['aug_mat'].shape

(11324, 11324)

In [61]:
adata_patient_organoid

AnnData object with n_obs × n_vars = 11324 × 20157
    obs: 'n_counts', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'original_total_counts', 'log10_original_total_counts', 'original_mito_counts', 'log10_original_mito_counts', 'Sample ID', 'emptyDrops_Total', 'emptyDrops_LogProb', 'emptyDrops_PValue', 'emptyDrops_FDR', 'latent_cell_probability', 'PhenoGraph_clusters', 'Smilie Cell Type', 'log10_n_counts', 'frac_counts_gt1_reads', 'n_reads', 'n_reads_per_count', 'n_counts_vs_ED_LogProb', 'n_counts_vs_n_genes_by_counts', 'n_flagged_metrics', 'DC 1', 'DC 2', 'DC 3', 'DC 4', 'DC 5', 'DC 6', 'DC 7', 'Palantir Diff. Potential', 'Palantir Pseudotime', 'Combined Fetal Signature', 'First Tri

In [67]:
import pandas as pd

# Assuming 'adata_patient_organoid' is your DataFrame
adata_patient_organoid.obs['PhenoGraph Classification'].value_counts()

PhenoGraph Classification
Enterocyte-like     3201
nan                 2232
TA-like             1913
ISC-like            1770
Injury Repair       1160
Goblet-like          371
Fetal Progenitor     318
SCC                  295
Early NET             64
Name: count, dtype: int64

In [13]:
adata_patient_organoid.obs.loc[adata_patient_organoid.obs['PhenoGraph Classification']=='nan']

,n_counts,n_genes,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,...,Module Endoderm Development Gene Score,Module Endoderm Development Score,Module Injury Repair Gene Score,Module Injury Repair Score,Module EMT Gene Score,Module EMT Score,Module Neuroendocrine Gene Score,Module Neuroendocrine Score,Media,Patient Tumor
120703424285939_KG146M,12791.0,4122,4122,8.324336,12791.0,9.456575,35.188805,40.575405,47.416152,58.603706,...,-0.144288,0.184177,-0.181300,-0.327645,-0.190323,-0.363053,-0.480979,-1.197567,Patient,Metastasis
120703436155741_KG146M,20657.0,4282,4282,8.362409,20657.0,9.935858,33.862613,42.358523,51.299802,64.660890,...,0.468292,-0.659373,1.274276,7.412497,0.243461,1.022820,0.392913,-1.109128,Patient,Metastasis
120703455025013_KG146M,38341.0,7117,7117,8.870382,38341.0,10.554301,36.214496,42.794919,49.894369,59.197726,...,0.185318,0.183288,-0.120878,0.751803,-0.178741,0.309788,-0.610722,-1.073547,Patient,Metastasis
120718456679846_KG146M,4448.0,1939,1939,7.570443,4448.0,8.400435,27.585432,37.230216,47.819245,64.568345,...,1.014287,2.839662,-0.191981,-1.736034,0.226650,3.434851,0.665020,5.820155,Patient,Metastasis
120718468987109_KG146M,9450.0,2948,2948,7.989221,9450.0,9.153876,37.947090,46.814815,55.544974,67.280423,...,-0.219792,-1.069259,0.038297,-0.127144,0.031821,-0.802285,-0.109984,-1.134468,Patient,Metastasis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165190304708846_KG146P,13465.0,4378,4378,8.384576,13465.0,9.507923,34.868177,39.339027,45.399183,56.212403,...,0.049669,-0.384234,-0.437856,0.500528,-0.236271,-1.078942,-0.562407,-1.102562,Patient,Primary
161384145836979_KG146P,18892.0,3967,3967,8.286017,18892.0,9.846547,52.186111,59.353165,65.286894,73.242642,...,-0.250045,-0.678173,-0.204253,-1.577774,-0.070241,-1.635169,-0.153235,-1.298806,Patient,Primary
200897607945508_KG146P,36858.0,5826,5826,8.670258,36858.0,10.514855,33.930219,47.346573,57.428509,67.293396,...,-0.450895,-1.372300,-0.287006,-1.480717,-0.159319,-1.152928,-0.304631,-1.321132,Patient,Primary
125033721486630_KG146P,43003.0,6076,6076,8.712266,43003.0,10.669048,38.018278,51.261540,59.712113,68.816129,...,-0.470417,-1.157184,-0.387183,-1.825860,-0.237220,-1.390900,-0.317327,-1.349161,Patient,Primary


In [18]:
train_adata.obs['Cell State']

120703408859571_OKG146P_Base             NA
120703423732149_OKG146P_Base             NA
120703423986909_OKG146P_Base             NA
120703436417774_OKG146P_Base             NA
120718456404405_OKG146P_Base             NA
                                   ...     
241109220584862_KG146M                  SCC
241176048225691_KG146M             ISC-like
241176061270774_KG146M          Goblet-like
241184516782309_KG146M                  SCC
241184517057252_KG146M              TA-like
Name: Cell State, Length: 10396, dtype: category
Categories (9, object): ['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', ..., 'Injury Repair', 'NA', 'SCC', 'TA-like']

In [29]:
tmp.loc[tmp.loc[:,'Cell State']=='NA','PhenoGraph Classification'].value_counts()

PhenoGraph Classification
Enterocyte-like     3201
TA-like             1913
ISC-like            1770
Injury Repair       1160
Goblet-like          371
Fetal Progenitor     318
SCC                  295
Early NET             64
nan                    0
Name: count, dtype: int64

In [30]:
tmp.loc[tmp.loc[:,'Cell State']=='nan','PhenoGraph Classification'].value_counts()

PhenoGraph Classification
nan                 928
Early NET             0
Enterocyte-like       0
Fetal Progenitor      0
Goblet-like           0
ISC-like              0
Injury Repair         0
SCC                   0
TA-like               0
Name: count, dtype: int64

In [5]:
# These are the cell type labels
adata_patient_organoid.obs['Cell State'].unique()

['NA', 'ISC-like', 'SCC', 'Fetal Progenitor', 'Enterocyte-like', 'TA-like', 'Injury Repair', 'Early NET', 'Goblet-like', 'nan']
Categories (10, object): ['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', ..., 'NA', 'SCC', 'TA-like', 'nan']

In [50]:
# Assuming adata_patient_organoid.obs is a DataFrame
cell_state_counts = adata_patient_organoid.obs['Cell State'].value_counts()

print(cell_state_counts)

Cell State
NA                  9092
nan                  928
SCC                  280
ISC-like             252
Injury Repair        230
Enterocyte-like      223
Fetal Progenitor     106
TA-like               80
Goblet-like           71
Early NET             62
Name: count, dtype: int64


In [57]:
adata_patient_organoid.obs['P(ISC-like)']

120703408859571_OKG146P_Base    0.276070
120703423732149_OKG146P_Base    0.235303
120703423986909_OKG146P_Base    0.233067
120703436417774_OKG146P_Base    0.139115
120718456404405_OKG146P_Base    0.256664
                                  ...   
165190304708846_KG146P               NaN
161384145836979_KG146P               NaN
200897607945508_KG146P               NaN
125033721486630_KG146P               NaN
200444891487139_KG146P               NaN
Name: P(ISC-like), Length: 11324, dtype: float64

In [20]:
labels.shape

(11324,)

In [21]:
data.shape

(11324, 20157)

In [22]:
unique_labels, counts = np.unique(labels, return_counts=True)
print("Unique Labels:", unique_labels)
print("Label Counts:", counts)

Unique Labels: [0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]
Label Counts: [ 928   62  223  106   71  252  230 9092  280   80]


In [26]:
def classify(train_adata, test_adata, k=30, metric='euclidean', n_jobs=-1):
    """
    Semi-supervised classification by random walks on a graph
    :param train_adata: AnnData object with labeled training data
    :param test_adata: AnnData object with unlabeled testing data
    :return c: class assignment for each row in test
    :return P: class probabilities for each row in test
    """
    # Extract labels and preprocess the data
    data, labels = preprocess(train_adata, test_adata)

    # Use augmented affinity matrix from adata_patient_organoid.obsm['aug_mat']
    A = adata_patient_organoid.obsm['aug_mat']

    # Ensure that the affinity matrix has the correct shape
    if A.shape[0] != A.shape[1] or A.shape[0] != len(labels):
        raise ValueError("Mismatch in the shape of the affinity matrix and labels.")

    # Compute random walk probabilities
    P = random_walk_probabilities(A, labels)

    # Class assignment based on the highest probability
    c = np.argmax(P, axis=1)

    return c, P

In [40]:
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import normalize

def random_walk_probabilities(A, labels):

    if labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()
        D = sp.diags(A.sum(axis=1).flatten(), [0], shape=A.shape, format='csr')
        L = D - A
        seeds = labels != 0
        Lu = L[~seeds, :][:, ~seeds]  # unlabeled rows, unlabeled cols
        Lu = Lu.tocsc()[:, ~seeds]  # unlabeled columns
        BT = L[~seeds, :][:, seeds]  # unlabeled rows, labeled cols
        BT = BT.tocsc()[:, seeds]  # labeled columns
        # Matrix representation of labels
        i, j, s = [], [], []
        classes = np.unique(labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(labels[seeds] == k)[0])
            j.extend(np.tile(int(k), sum(seeds == k)))
            s.extend(np.tile(1, sum(seeds == k)))
        i = np.arange(seeds.sum())
        j = labels[seeds] - 1
        s = np.ones((seeds.sum(), ))
        M = sp.coo_matrix((s, (i, j)), shape=(seeds.sum(), len(classes))).tocsc()
        # P = sp.linalg.spsolve(Lu.tocsc(), -BT.dot(M))
        # Use iterative solver
        B = -BT.dot(M)
        vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]
        warnings = [x[1] for x in vals]
        if any(warnings):
            print("Warning: iterative solver failed to converge in at least one case", flush=True)
        P = normalize(np.vstack(tuple((x[0] for x in vals))).T, norm='l1')
        
    else:
        D = np.diag(np.sum(A, axis=1))
        L = D - A  # graph laplacian
        seeds = np.array([e != 0 for e in labels], dtype=bool)
        Lu = L[~seeds, :][:, ~seeds]  # unlabeled rows, unlabeled cols
        BT = L[~seeds, :][:, seeds]  # unlabeled rows, labeled cols
        classes = np.unique(labels[labels > 0])
        M = np.zeros((seeds.sum(), len(classes)))
        for k in classes:
            if k in labels[seeds]:
                M[labels[seeds] == k, np.where(classes == k)[0][0]] = 1
        P = np.linalg.lstsq(Lu, np.dot(-BT, M), rcond=None)[0]


    return P


In [41]:
c, P = classify(train_adata, test_adata, k=30, metric='euclidean', n_jobs=-1)

In [45]:
c.shape

(928,)

In [46]:
P.shape

(928, 9)

In [47]:
# Save class assignments (c) and probabilities (P)
np.savez('/data/chanjlab/forsythb/phenograph_classification_results.npz', c=c, P=P)

In [ ]:
# Extract patient and organoid data
# The patient data is labeled, so it is the training data
# The organoid data is not labeled, so it is the testing data

patient_data = adata_patient_organoid[adata_patient_organoid.obs['Media'] == 'Patient'].copy()
organoid_data = adata_patient_organoid[adata_patient_organoid.obs['Media'].isin(['Base', 'hISC'])].copy()

In [ ]:
patient_data.obs['Cell State']

In [ ]:
organoid_data.obs['Cell State']

In [ ]:
train = patient_data[:5]
test = organoid_data[:5]

In [ ]:
def preprocess(train, test):
    labels = np.zeros((test.shape[0], ), dtype=int)
    data = test
    for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data = np.append(data, examples, axis=0)
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    return data, labels

In [ ]:
labels = np.zeros((test.shape[0], ), dtype=int)
labels

In [ ]:
data = test

In [ ]:
for c, examples in enumerate(train):
    labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)

In [ ]:
labels

In [ ]:
preprocess(train, test)

In [ ]:
for c, examples in enumerate(train[:2]):  # Limiting to 2 iterations for illustration
        # Step 4: Append class labels to labels array
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        
        # Step 5: Append examples to data array
        data = np.append(data, examples, axis=0)

In [ ]:
train = combined_data[combined_data.obs['Media'] == 'Patient'].copy()
test = combined_data[combined_data.obs['Media'].isin(['Base', 'hISC'])].copy()

In [ ]:
labels = np.zeros((test.shape[0], ), dtype=int)
data = test
labels.shape
data.shape

In [ ]:
for c, examples in enumerate(train):
    labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)

In [ ]:
labels

In [ ]:
import numpy as np
import pandas as pd
import anndata

# Set a seed for reproducibility
np.random.seed(42)

# Extract patient and organoid data
patient_data = adata_patient_organoid[adata_patient_organoid.obs['Media'] == 'Patient'].copy()
organoid_data = adata_patient_organoid[adata_patient_organoid.obs['Media'].isin(['Base', 'hISC'])].copy()

# Randomly choose 50 samples from each subset
random_patient_indices = np.random.choice(patient_data.n_obs, size=50, replace=False)
random_organoid_indices = np.random.choice(organoid_data.n_obs, size=50, replace=False)

# Subset the data using random indices
random_patient_data = patient_data[random_patient_indices].copy()
random_organoid_data = organoid_data[random_organoid_indices].copy()

# Combine the subsets into one AnnData object
combined_data = anndata.concat([random_patient_data, random_organoid_data])

In [ ]:
train = combined_data[combined_data.obs['Media'] == 'Patient'].copy()
test = combined_data[combined_data.obs['Media'].isin(['Base', 'hISC'])].copy()

In [ ]:
labels = np.zeros((test.shape[0], ), dtype=int)
labels.shape

In [ ]:
data_list = [test]

In [ ]:
import anndata

In [ ]:
for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data_list.append(examples)

# Concatenate the list of AnnData objects along the observation axis
data = anndata.concat(data_list, index_unique=None)

In [ ]:
data

In [ ]:
data = data
labels = labels
def classify(train, test, k=30, metric='euclidean', n_jobs=-1):
    """
    Semi-supervised classification by random walks on a graph
    :param train: list of numpy arrays. Each array has a row for each class observation
    :param test: numpy array of unclassified data
    :return c: class assignment for each row in test
    :return P: class probabilities for each row in test
    """
    # Build graph
    A = combined_data.obsm['aug_mat']
    P = random_walk_probabilities(A, labels)
    c = np.argmax(P, axis=1)
    return c, P

In [ ]:
def random_walk_probabilities(A, labels):

    if labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()

        # Create a diagonal matrix using the sum of each row in A
        D = sp.diags(A.sum(axis=1).A1, [0], shape=A.shape, format='csr')

        L = D - A
        seeds = labels != 0
        Lu = L[~seeds, :][:, ~seeds]
        BT = L[~seeds, :][:, seeds]

        # Matrix representation of labels
        i, j, s = [], [], []
        classes = np.unique(labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(labels[seeds] == k)[0])
            j.extend(np.tile(k, np.sum(seeds == k)))
            s.extend(np.tile(1, np.sum(seeds == k)))
        i = np.arange(np.sum(seeds))
        j = labels[seeds] - 1
        s = np.ones((np.sum(seeds), ))
        M = sp.coo_matrix((s, (i, j)), shape=(np.sum(seeds), len(classes))).tocsc()

        # Solve the linear system using an iterative solver
        P = normalize(sp.linalg.bicgstab(Lu, -BT.dot(M))[0].reshape(-1, 1), norm='l1', axis=1)

    else:
        D = np.diag(np.sum(A, axis=1))
        L = D - A
        seeds = np.array([e != 0 for e in labels], dtype=bool)
        Lu = L[np.invert(seeds), :][:, np.invert(seeds)]
        BT = L[np.invert(seeds), :][:, seeds]

        # Matrix representation of labels
        classes = np.unique(labels[labels > 0])
        M = np.zeros((np.sum(seeds), len(classes)))
        for k in classes:
            M[labels[seeds] == k, k] = 1

        P = normalize(np.linalg.lstsq(Lu, np.dot(-BT, M))[0], norm='l1', axis=1)

    return P


In [ ]:
classify(train, test, k=30, metric='euclidean', n_jobs=-1)

In [ ]:
cpn

In [ ]:
'''
Subset adata to include only the patient data 
(This is 'Patient' in .obs['Media'])
'''
adata_patient = adata_patient_organoid[adata_patient_organoid.obs['Media'] == 'Patient'].copy()

# Select 100 random cells
random_indices = np.random.choice(adata_patient.n_obs, size=100, replace=False)
adata_patient_subset = adata_patient[random_indices].copy()

In [ ]:
'''
Subset adata to include only the organoid data
(This is 'Base' and 'hISC' in .obs['Media'])
''' 
adata_organoid = adata_patient_organoid[adata_patient_organoid.obs['Media'].isin(['Base', 'hISC'])].copy()

# Select 100 random cells
random_indices = np.random.choice(adata_organoid.n_obs, size=100, replace=False)
adata_organoid_subset = adata_organoid[random_indices].copy()

In [ ]:
cell_types = np.sort(adata_patient_organoid.obs['Cell State'].unique())
print(cell_types)

In [ ]:
dm_merge = pd.DataFrame(adata_patient_organoid.obsm['aug_mat'], index=adata_patient_organoid.obs.index)
dm_merge

In [ ]:
train = np.empty((len(cell_types),),dtype=object)
train

In [ ]:
for c,cell_type in enumerate(cell_types):
    labels = adata_patient_organoid.obs.index[adata_patient_organoid.obs['Cell State'] == cell_type]
    ind = [adata_patient_organoid.obs.index.get_loc(i) for i in labels]
    train[c] = dm_merge.iloc[ind,:]

In [ ]:
test = dm_merge

In [ ]:
for c, cell_type in enumerate(cell_types):
    labels = adata_patient_organoid.obs.index[adata_patient_organoid.obs['Cell State'] == cell_type]
    ind = [adata_patient_organoid.obs.index.get_loc(i) for i in labels]
    train[c] = dm_merge.iloc[ind, :]
    print(f"Cell type: {cell_type}, Indices: {ind}")

# Check the length of train before accessing index 10338
if 10338 < len(train):
    print(train[10338])
else:
    print(f"Index 10338 is out of bounds for train with length {len(train)}")

